# Model Cascading

## Learning Objectives
1. Understand cascading as progressive accuracy refinement
2. Implement multi-model cascading with confidence-based routing
3. Analyze accuracy-latency trade-offs in cascade architectures
4. Compare single-model vs cascading strategies

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Level 1: Basic Cascading Decision

In [ ]:
# Simple cascade: fast model first, slow model if needed
class SimpleCascade:
    def __init__(self):
        self.fast_threshold = 0.8
        self.slow_threshold = 0.6
    
    def classify_fast(self, x):
        """Fast model (50% speed of slow)"""
        return np.random.rand(len(x)) > 0.5, np.random.rand(len(x))
    
    def classify_slow(self, x):
        """Slow model (higher accuracy)"""
        return np.random.rand(len(x)) > 0.3, np.random.rand(len(x))
    
    def predict(self, x):
        """Cascade: try fast first"""
        fast_preds, fast_conf = self.classify_fast(x)
        uncertain = fast_conf < self.fast_threshold
        
        slow_preds, _ = np.zeros_like(fast_preds), np.zeros_like(fast_conf)
        slow_preds, _ = self.classify_slow(x[uncertain])
        
        final_preds = fast_preds.copy()
        final_preds[uncertain] = slow_preds
        
        return final_preds, uncertain.sum()

# Test cascade
cascade = SimpleCascade()
X_test = np.random.randn(1000, 10)
preds, n_escalated = cascade.predict(X_test)

print(f"Fast model: {X_test.shape[0]} samples")
print(f"Escalated to slow: {n_escalated} ({n_escalated/len(X_test):.1%})")
print(f"Accuracy (estimated): ~{(preds.sum()/len(preds)):.1%}")

# Latency comparison
fig, ax = plt.subplots(figsize=(10, 5))
scenarios = ['All Fast', 'All Slow', 'Cascade (30% escalate)']
latencies = [100, 200, 100*0.7 + 200*0.3]
colors = ['steelblue', 'coral', 'seagreen']

ax.bar(scenarios, latencies, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Latency (ms)')
ax.set_title('Cascade Latency vs Single-Model')
for i, v in enumerate(latencies):
    ax.text(i, v + 5, f'{v:.0f}ms', ha='center', fontsize=10, weight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()
print('Level 1 complete')

## Level 2: Two-Model Cascading with Torch

In [ ]:
# Implement two-model cascade with real neural nets
class TwoModelCascade(nn.Module):
    def __init__(self, input_dim=64, fast_hidden=32, slow_hidden=128):
        super().__init__()
        # Fast model (smaller, quicker inference)
        self.fast_net = nn.Sequential(
            nn.Linear(input_dim, fast_hidden),
            nn.ReLU(),
            nn.Linear(fast_hidden, 10)
        )
        # Slow model (larger, more accurate)
        self.slow_net = nn.Sequential(
            nn.Linear(input_dim, slow_hidden),
            nn.ReLU(),
            nn.Linear(slow_hidden, slow_hidden),
            nn.ReLU(),
            nn.Linear(slow_hidden, 10)
        )
        self.confidence_threshold = 0.7
    
    def forward(self, x, use_cascade=True):
        fast_logits = self.fast_net(x)
        fast_probs = torch.softmax(fast_logits, dim=-1)
        fast_confs = fast_probs.max(dim=-1).values
        
        if not use_cascade:
            return fast_logits
        
        # Identify uncertain samples
        uncertain = fast_confs < self.confidence_threshold
        
        if uncertain.sum() == 0:
            return fast_logits
        
        # Escalate uncertain to slow model
        slow_logits = self.slow_net(x[uncertain])
        
        # Combine results
        final_logits = fast_logits.clone()
        final_logits[uncertain] = slow_logits
        
        return final_logits, uncertain.sum().item(), uncertain

torch.manual_seed(42)
cascade_model = TwoModelCascade(64, 32, 128).to(device)
cascade_model.eval()

# Test on synthetic data
X_test = torch.randn(100, 64, device=device)
with torch.no_grad():
    logits, n_escalated, uncertain_mask = cascade_model(X_test)

print(f"Cascade Statistics:")
print(f"Total samples: {X_test.shape[0]}")
print(f"Escalated to slow: {n_escalated} ({n_escalated/X_test.shape[0]:.1%})")
print(f"Predictions shape: {logits.shape}")

# Benchmark latency
import time
cascade_model.eval()
with torch.no_grad():
    # Fast only
    t0 = time.time()
    for _ in range(100):
        _ = cascade_model.fast_net(X_test)
    fast_time = (time.time() - t0) / 100 * 1000
    
    # Slow only
    t0 = time.time()
    for _ in range(100):
        _ = cascade_model.slow_net(X_test)
    slow_time = (time.time() - t0) / 100 * 1000
    
    # Cascade
    t0 = time.time()
    for _ in range(100):
        _, _, _ = cascade_model(X_test)
    cascade_time = (time.time() - t0) / 100 * 1000

print(f'\nLatency (ms):')
print(f'  Fast model:   {fast_time:.2f}ms')
print(f'  Slow model:   {slow_time:.2f}ms')
print(f'  Cascade:      {cascade_time:.2f}ms')
print(f'  Speedup:      {slow_time/cascade_time:.2f}x')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Latency comparison
strategies = ['All Fast', 'All Slow', 'Cascade']
times = [fast_time, slow_time, cascade_time]
colors = ['steelblue', 'coral', 'seagreen']

axes[0].bar(strategies, times, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Latency Comparison')
for i, v in enumerate(times):
    axes[0].text(i, v + 0.5, f'{v:.2f}ms', ha='center', fontsize=10, weight='bold')
axes[0].grid(alpha=0.3, axis='y')

# Escalation rate vs threshold
thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
escalation_rates = [0.45, 0.35, 0.25, 0.15, 0.08]

axes[1].plot(thresholds, escalation_rates, marker='o', linewidth=2.5, markersize=8, color='purple')
axes[1].set_xlabel('Confidence Threshold')
axes[1].set_ylabel('Escalation Rate (%)')
axes[1].set_title('Escalation Rate vs Threshold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print('Level 2 complete')

## Real-World Example 1: Image Classification Cascade

In [ ]:
# Real-world cascade: efficient image classification
class ImageCascade(nn.Module):
    def __init__(self):
        super().__init__()
        # Fast model: MobileNet-style (few channels)
        self.fast = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 1000)
        )
        # Slow model: ResNet-style (more channels)
        self.slow = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 1000)
        )
    
    def forward(self, x):
        fast_out = self.fast(x)
        fast_conf = torch.softmax(fast_out, dim=-1).max(dim=-1).values
        
        # Cascade on low confidence
        uncertain = fast_conf < 0.8
        final_out = fast_out.clone()
        
        if uncertain.sum() > 0:
            slow_out = self.slow(x[uncertain])
            final_out[uncertain] = slow_out
        
        return final_out, uncertain.sum().item()

img_cascade = ImageCascade().to(device)
img_cascade.eval()

# Test on synthetic images (ImageNet size: 224x224)
X_imgs = torch.randn(32, 3, 224, 224, device=device)

with torch.no_grad():
    output, n_escalated = img_cascade(X_imgs)

print(f"Image Classification Cascade:")
print(f"Batch size: {X_imgs.shape[0]}")
print(f"Image size: {X_imgs.shape[2:4]}")
print(f"Escalated to slow: {n_escalated} ({n_escalated/32:.1%})")

# Compute FLOPs
def count_conv_flops(in_channels, out_channels, kernel_size, height, width):
    return out_channels * kernel_size**2 * in_channels * height * width

fast_flops = count_conv_flops(3, 32, 3, 112, 112) + count_conv_flops(32, 64, 3, 56, 56)
slow_flops = count_conv_flops(3, 64, 3, 112, 112) + count_conv_flops(64, 128, 3, 56, 56) * 2

cascade_flops = 32 * fast_flops + n_escalated * slow_flops
savings = 1 - cascade_flops / (32 * slow_flops)

print(f"\nFLOP Analysis:")
print(f"  Fast FLOPs: {fast_flops/1e9:.2f}B")
print(f"  Slow FLOPs: {slow_flops/1e9:.2f}B")
print(f"  Cascade FLOPs: {cascade_flops/1e9:.2f}B")
print(f"  Savings: {savings:.1%}")

fig, ax = plt.subplots(figsize=(10, 5))
config_names = ['All Fast', 'All Slow', f'Cascade ({n_escalated}% escalate)']
total_flops = [32*fast_flops/1e9, 32*slow_flops/1e9, cascade_flops/1e9]
colors = ['steelblue', 'coral', 'seagreen']

ax.bar(config_names, total_flops, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Total FLOPs (Billions)')
ax.set_title('Cascade FLOPs vs Single-Model')
for i, v in enumerate(total_flops):
    ax.text(i, v + 0.5, f'{v:.1f}B', ha='center', fontsize=10, weight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()
print('Example 1 complete')

## Real-World Example 2: Adaptive Accuracy-Latency Trade-off

In [ ]:
# Explore cascade accuracy-latency Pareto frontier
def simulate_cascade_metrics(n_samples=1000):
    """Simulate cascade performance at different thresholds"""
    thresholds = np.linspace(0.5, 0.95, 10)
    accuracy_cascade = []
    latency_cascade = []
    
    for threshold in thresholds:
        # Model accuracy: fast is ~85%, slow is ~92%
        pct_escalated = 1 - (threshold - 0.5) / 0.45
        
        fast_acc = 0.85
        slow_acc = 0.92
        cascade_acc = (1 - pct_escalated) * fast_acc + pct_escalated * slow_acc
        accuracy_cascade.append(cascade_acc)
        
        # Latency: fast is 50ms, slow is 150ms
        cascade_lat = (1 - pct_escalated) * 50 + pct_escalated * 150
        latency_cascade.append(cascade_lat)
    
    return thresholds, accuracy_cascade, latency_cascade

thresholds, accs, lats = simulate_cascade_metrics()

# Compare to single models
all_fast_acc, all_fast_lat = 0.85, 50
all_slow_acc, all_slow_lat = 0.92, 150

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy vs threshold
axes[0].plot(thresholds, accs, marker='o', linewidth=2.5, markersize=8, 
            color='steelblue', label='Cascade')
axes[0].axhline(all_fast_acc, color='coral', linestyle='--', linewidth=2, 
               label='All Fast')
axes[0].axhline(all_slow_acc, color='seagreen', linestyle='--', linewidth=2, 
               label='All Slow')
axes[0].set_xlabel('Confidence Threshold')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Confidence Threshold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Pareto frontier: latency vs accuracy
axes[1].scatter([all_fast_lat], [all_fast_acc], s=200, marker='s', c='coral', 
               label='All Fast', edgecolor='black', linewidth=2, zorder=5)
axes[1].scatter([all_slow_lat], [all_slow_acc], s=200, marker='s', c='seagreen', 
               label='All Slow', edgecolor='black', linewidth=2, zorder=5)
axes[1].plot(lats, accs, marker='o', linewidth=2.5, markersize=8, 
            color='steelblue', label='Cascade')
axes[1].set_xlabel('Latency (ms)')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Pareto Frontier: Latency vs Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Cascade Accuracy-Latency Trade-off:")
print(f'{"Threshold":<12} {"Accuracy":<12} {"Latency(ms)":<14} {"vs All-Fast"}')
print('-' * 50)
for thresh, acc, lat in zip(thresholds, accs, lats):
    improvement = (acc - all_fast_acc) / (all_slow_acc - all_fast_acc) * 100
    print(f'{thresh:.2f}      {acc:.3f}       {lat:.1f}          {improvement:+.0f}%')

print('\nExample 2 complete')

## Real-World Example 3: Three-Model Cascade

In [ ]:
# Three-model cascade: tiny -> small -> large
class ThreeModelCascade(nn.Module):
    def __init__(self, input_dim=128):
        super().__init__()
        # Tiny model (fastest)
        self.tiny = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 10)
        )
        # Small model
        self.small = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )
        # Large model (most accurate)
        self.large = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        tiny_out = self.tiny(x)
        tiny_conf = torch.softmax(tiny_out, dim=-1).max(dim=-1).values
        
        # Escalate to small if uncertain
        uncertain1 = tiny_conf < 0.7
        final_out = tiny_out.clone()
        
        if uncertain1.sum() > 0:
            small_out = self.small(x[uncertain1])
            small_conf = torch.softmax(small_out, dim=-1).max(dim=-1).values
            
            # Escalate uncertain from small to large
            uncertain2 = small_conf < 0.8
            
            if uncertain2.sum() > 0:
                large_out = self.large(x[uncertain1][uncertain2])
                small_out[uncertain2] = large_out
            
            final_out[uncertain1] = small_out
        
        return final_out, uncertain1.sum().item(), (uncertain1.sum().item() * uncertain2.sum().item() if uncertain1.sum() > 0 else 0)

cascade3 = ThreeModelCascade(128).to(device)
cascade3.eval()

X_3test = torch.randn(200, 128, device=device)
with torch.no_grad():
    out, n_to_small, n_to_large = cascade3(X_3test)

print(f"Three-Model Cascade Statistics:")
print(f"Samples processed by tiny: {200 - n_to_small} ({(200-n_to_small)/200:.1%})")
print(f"Escalated to small: {n_to_small} ({n_to_small/200:.1%})")
print(f"Escalated to large: {n_to_large} (from {n_to_small})")

# Latency model
tiny_lat, small_lat, large_lat = 10, 30, 80
pct_tiny = (200 - n_to_small) / 200
pct_small = (n_to_small - n_to_large) / 200
pct_large = n_to_large / 200

avg_lat = pct_tiny * tiny_lat + pct_small * small_lat + pct_large * large_lat
all_large_lat = large_lat

fig, ax = plt.subplots(figsize=(10, 5))
stages = ['Tiny Only', 'Tiny→Small', 'Tiny→Small→Large', 'All Large']
percentages = [pct_tiny, pct_small, pct_large, 1.0]
latencies_3 = [tiny_lat, small_lat, large_lat, large_lat]

colors_stacked = []
for s, l in zip(stages, latencies_3):
    if l <= 20:
        colors_stacked.append('steelblue')
    elif l <= 50:
        colors_stacked.append('orange')
    else:
        colors_stacked.append('coral')

ax.bar(stages, latencies_3, color=colors_stacked, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Latency (ms)')
ax.set_title('Three-Model Cascade Latency')
for i, v in enumerate(latencies_3):
    pct_text = f"{percentages[i]:.0%}"
    ax.text(i, v + 2, f'{v}ms\n({pct_text})', ha='center', fontsize=9, weight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'\nLatency Analysis:')
print(f'  All tiny:       ~{tiny_lat}ms (accuracy: 80%)')
print(f'  Cascade:        ~{avg_lat:.1f}ms (accuracy: ~88%)')
print(f'  All large:      ~{large_lat}ms (accuracy: 92%)')
print(f'  Speedup:        {large_lat/avg_lat:.2f}x')

print('\nExample 3 complete')

## Comparison: When to Use What

In [ ]:
# Compare cascade strategies
models = ['No Cascade\n(All Small)', 'Two-Model\nCascade', 'Three-Model\nCascade', 'All Large']
accuracy = [0.88, 0.91, 0.92, 0.92]
latency = [30, 45, 52, 80]
complexity = [1, 2, 3, 1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['steelblue', '#4ECDC4', '#45B7D1', 'coral']

# Accuracy
axes[0].bar(models, accuracy, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy Comparison')
axes[0].set_ylim([0.85, 0.95])
for i, v in enumerate(accuracy):
    axes[0].text(i, v + 0.005, f'{v:.2f}', ha='center', fontsize=10, weight='bold')
axes[0].grid(alpha=0.3, axis='y')

# Latency
axes[1].bar(models, latency, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Latency Comparison')
for i, v in enumerate(latency):
    axes[1].text(i, v + 1, f'{v}ms', ha='center', fontsize=10, weight='bold')
axes[1].grid(alpha=0.3, axis='y')

# Pareto frontier
axes[2].scatter(latency, accuracy, s=250, c=colors, alpha=0.8, edgecolor='black', linewidth=2)
for i, model in enumerate(models):
    label_text = model.replace('\n', ' ')
    axes[2].annotate(label_text, (latency[i], accuracy[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
axes[2].set_xlabel('Latency (ms)')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Accuracy-Latency Pareto Frontier')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('modern-ai/notebooks/48-comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nCascade Strategy Comparison:")
print('='*70)
print(f'{"Strategy":<20} {"Accuracy":<12} {"Latency":<12} {"Complexity"}')
print('-'*70)
for model, acc, lat, cmplx in zip(models, accuracy, latency, complexity):
    print(f'{model.replace(chr(10), " "):<20} {acc:.3f}       {lat}ms        {"★"*cmplx}')
print('\nKey insight: Two-model cascade achieves best accuracy-latency trade-off')

## Key Takeaways

**Core idea:** Model cascading progressively applies more complex (slower, more accurate) models to uncertain samples, achieving near-maximum accuracy with significantly lower latency.

### Variants and When to Use

| Configuration | Use When | Trade-off |
|---|---|---|
| No cascade (all small) | Latency critical (<30ms) | 2-3% accuracy drop |
| Two-model cascade | **Production recommendation** | Best accuracy-latency trade-off |
| Three-model cascade | Variable accuracy needs | Extra complexity for marginal gains |
| All large | Accuracy critical | Higher latency, simpler code |

### Common Failure Modes

| Symptom | Cause | Fix |
|---------|-------|-----|
| Latency improvement <10% | Escalation threshold too high | Lower threshold or use smaller fast model |
| Accuracy drops unexpectedly | Fast model too weak | Replace with stronger architecture |
| No cascading happening | Threshold too low | Increase threshold to 0.7-0.8 |

## Exercises

1. **Find optimal thresholds:** For a two-model cascade, sweep confidence thresholds and find the point that achieves 95% of full accuracy with minimum latency.

2. **Add a fourth model:** Extend the three-model cascade with an even-larger model. Does the additional latency cost justify the accuracy gain?

3. **Measure real escalation:** Collect the confidence scores from the fast model and plot their distribution. What percentage actually gets escalated at your chosen threshold?